# Week 7 Assignment
# Document Question Answering System using RAG

## Name: Rishabh Sah

### Objective
To build a Retrieval-Augmented Generation (RAG) system that answers questions from custom documents.

### Workflow
1. Load PDF document
2. Split text into chunks
3. Create embeddings
4. Store embeddings in FAISS
5. Retrieve relevant chunks
6. Generate answers using a local FLAN-T5 model

In [1]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-huggingface
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 11.5 MB/s eta 0:00:00


In [2]:
import os

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS



/tmp/ipykernel_907/1958247053.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
uploaded = files.upload()

Saving Rishabh Sah_PCE23AD042 (2).pdf to Rishabh Sah_PCE23AD042 (2).pdf


In [4]:
loader = PyPDFLoader("Rishabh Sah_PCE23AD042 (2).pdf")

documents = loader.load()

print("Total Pages Loaded:", len(documents))

Total Pages Loaded: 1


## Text Chunking

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 4


In [6]:
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1}\n")
    print(chunk.page_content)
    print("-"*80)


Chunk 1

RISHABH SAH
Software Developer
7981132525 — rishabhsah143@gmail.com
linkedin — github
SUMMARY
Aspiring Software Developer with strong foundations in programming, OOPs, and core data structures. Skilled in
Python and C++ with experience in building applications and working with MySQL.
COURSEWORK / SKILLS
•Data Structures & Algorithms
•Artificial Intelligence
•Machine Learning
•DataBase Management System (DBMS)
•Web Development
•OOPS Concept
•Networking System
•Operating Systems
TECHNICAL SKILLS
--------------------------------------------------------------------------------

Chunk 2

•Operating Systems
TECHNICAL SKILLS
Languages:C++, Python, HTML, CSS
Database:MySQL
T ools:Git, GitHub, VS Code
Concepts:OOPs, API Integration, Problem Solving
PROJECTS
NeoNest – AI-Enabled Healthcare Platform
HTML, CSS, Python /github/external-link-alt
•Developed responsive web application for healthcare services
•Implemented user workflows and onboarding
•Designed frontend with responsive layout

## Create Embeddings

In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded Successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded Successfully


## Create Vector Database using FAISS

In [8]:
vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS Vector Store Created Successfully")

FAISS Vector Store Created Successfully


## Configure Local Language Model (FLAN-T5)

In [9]:
!pip install -q transformers torch accelerate

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

print("Local Language Model Loaded Successfully")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Local Language Model Loaded Successfully


## Retrieval and Question Answering Function

In [11]:
def ask_question(query):

    retriever = vector_db.as_retriever(search_kwargs={"k": 3})

    docs = retriever.invoke(query)

    print("\nRetrieved Context:\n")
    print("=" * 100)

    context = ""

    for i, doc in enumerate(docs):
        print(f"\nChunk {i+1}:\n")
        print(doc.page_content)
        print("-" * 100)

        context += doc.page_content + "\n"

    prompt = f"""
Answer the question only from the provided context.

Context:
{context}

Question:
{query}

If the answer is not present in the context, say:
"Answer not found in document."

Answer:
"""

    print("\nGenerating Answer...\n")

    try:
        inputs = flan_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        output_ids = flan_model.generate(**inputs, max_new_tokens=100)
        answer = flan_tokenizer.decode(output_ids[0], skip_special_tokens=True)

        print("\nGenerated Answer:\n")
        print("=" * 100)

        print(answer)

    except Exception as e:
        print("Error while generating answer:")
        print(e)

## Test Questions

In [12]:
ask_question("What projects are mentioned in the document?")


Retrieved Context:


Chunk 1:

•Operating Systems
TECHNICAL SKILLS
Languages:C++, Python, HTML, CSS
Database:MySQL
T ools:Git, GitHub, VS Code
Concepts:OOPs, API Integration, Problem Solving
PROJECTS
NeoNest – AI-Enabled Healthcare Platform
HTML, CSS, Python /github/external-link-alt
•Developed responsive web application for healthcare services
•Implemented user workflows and onboarding
•Designed frontend with responsive layout
LLM Hallucination Auditor – AI V erification System
Python, NLP, NumPy, Pandas
----------------------------------------------------------------------------------------------------

Chunk 2:

Python, NLP, NumPy, Pandas
•Built system to validate AI-generated responses
•Implemented scoring logic and modular design
•Optimized data processing workflows
INTERNSHIP
CREA TIX Pvt. Ltd. 2025
Summer Intern
•Developed AI chatbot and implemented TF-IDF search
•Integrated Google Gemini API
EDUCATION
B.T ech – Artificial Intelligence & Data ScienceCGPA: 8
Poornima College of 

In [13]:
ask_question("What technical skills are mentioned?")


Retrieved Context:


Chunk 1:

•Operating Systems
TECHNICAL SKILLS
Languages:C++, Python, HTML, CSS
Database:MySQL
T ools:Git, GitHub, VS Code
Concepts:OOPs, API Integration, Problem Solving
PROJECTS
NeoNest – AI-Enabled Healthcare Platform
HTML, CSS, Python /github/external-link-alt
•Developed responsive web application for healthcare services
•Implemented user workflows and onboarding
•Designed frontend with responsive layout
LLM Hallucination Auditor – AI V erification System
Python, NLP, NumPy, Pandas
----------------------------------------------------------------------------------------------------

Chunk 2:

Python, NLP, NumPy, Pandas
•Built system to validate AI-generated responses
•Implemented scoring logic and modular design
•Optimized data processing workflows
INTERNSHIP
CREA TIX Pvt. Ltd. 2025
Summer Intern
•Developed AI chatbot and implemented TF-IDF search
•Integrated Google Gemini API
EDUCATION
B.T ech – Artificial Intelligence & Data ScienceCGPA: 8
Poornima College of 

In [14]:
ask_question("What is the educational qualification?")


Retrieved Context:


Chunk 1:

Higher Secondary Education 83.3%
ACHIEVEMENTS
•Top 8 Finalist – Smart Ideathon 2K25 (2300+ teams)/external-link-alt
CERTIFICATIONS
•Google Cloud Skill Badge – AI Applications with Gemini & Imagen
•Participated in Code Slayer (NIT Delhi), HackWithMAIT, ISRO Hackathon/external-link-alt
----------------------------------------------------------------------------------------------------

Chunk 2:

Python, NLP, NumPy, Pandas
•Built system to validate AI-generated responses
•Implemented scoring logic and modular design
•Optimized data processing workflows
INTERNSHIP
CREA TIX Pvt. Ltd. 2025
Summer Intern
•Developed AI chatbot and implemented TF-IDF search
•Integrated Google Gemini API
EDUCATION
B.T ech – Artificial Intelligence & Data ScienceCGPA: 8
Poornima College of Engineering, Jaipur
Higher Secondary Education 83.3%
ACHIEVEMENTS
----------------------------------------------------------------------------------------------------

Chunk 3:

RISHABH SAH
Softw

In [15]:
ask_question("Summarize the document.")


Retrieved Context:


Chunk 1:

•Operating Systems
TECHNICAL SKILLS
Languages:C++, Python, HTML, CSS
Database:MySQL
T ools:Git, GitHub, VS Code
Concepts:OOPs, API Integration, Problem Solving
PROJECTS
NeoNest – AI-Enabled Healthcare Platform
HTML, CSS, Python /github/external-link-alt
•Developed responsive web application for healthcare services
•Implemented user workflows and onboarding
•Designed frontend with responsive layout
LLM Hallucination Auditor – AI V erification System
Python, NLP, NumPy, Pandas
----------------------------------------------------------------------------------------------------

Chunk 2:

Python, NLP, NumPy, Pandas
•Built system to validate AI-generated responses
•Implemented scoring logic and modular design
•Optimized data processing workflows
INTERNSHIP
CREA TIX Pvt. Ltd. 2025
Summer Intern
•Developed AI chatbot and implemented TF-IDF search
•Integrated Google Gemini API
EDUCATION
B.T ech – Artificial Intelligence & Data ScienceCGPA: 8
Poornima College of 

## System Metrics Report

In [16]:
print("===== SYSTEM METRICS =====")

print("Total Documents:", len(documents))
print("Total Chunks:", len(chunks))
print("Chunk Size:", 500)
print("Chunk Overlap:", 50)

print("Embedding Model: all-MiniLM-L6-v2")
print("Embedding Dimension: 384")

print("Vector Database: FAISS")
print("Retriever Top-K:", 3)

===== SYSTEM METRICS =====
Total Documents: 1
Total Chunks: 4
Chunk Size: 500
Chunk Overlap: 50
Embedding Model: all-MiniLM-L6-v2
Embedding Dimension: 384
Vector Database: FAISS
Retriever Top-K: 3


## Experiment: Comparing Chunk Sizes (Optimization Task)

In [17]:
small_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)

small_chunks = small_splitter.split_documents(documents)
small_vector_db = FAISS.from_documents(small_chunks, embeddings)

print("Chunk size 500 -> Total Chunks:", len(chunks))
print("Chunk size 200 -> Total Chunks:", len(small_chunks))

test_query = "What technical skills are mentioned?"

print("\n--- Retrieved with chunk_size=500 ---")
for doc in vector_db.as_retriever(search_kwargs={"k": 3}).invoke(test_query):
    print(doc.page_content[:150], "...\n")

print("\n--- Retrieved with chunk_size=200 ---")
for doc in small_vector_db.as_retriever(search_kwargs={"k": 3}).invoke(test_query):
    print(doc.page_content[:150], "...\n")

Chunk size 500 -> Total Chunks: 4
Chunk size 200 -> Total Chunks: 10

--- Retrieved with chunk_size=500 ---
•Operating Systems
TECHNICAL SKILLS
Languages:C++, Python, HTML, CSS
Database:MySQL
T ools:Git, GitHub, VS Code
Concepts:OOPs, API Integration, Proble ...

Python, NLP, NumPy, Pandas
•Built system to validate AI-generated responses
•Implemented scoring logic and modular design
•Optimized data processing w ...

RISHABH SAH
Software Developer
7981132525 — rishabhsah143@gmail.com
linkedin — github
SUMMARY
Aspiring Software Developer with strong foundations in p ...


--- Retrieved with chunk_size=200 ---
•Machine Learning
•DataBase Management System (DBMS)
•Web Development
•OOPS Concept
•Networking System
•Operating Systems
TECHNICAL SKILLS
Languages:C ...

Python, NLP, NumPy, Pandas
•Built system to validate AI-generated responses
•Implemented scoring logic and modular design
•Optimized data processing w ...

CREA TIX Pvt. Ltd. 2025
Summer Intern
•Developed AI chatbot and implement

# Experiment and Observations

1. Chunk size of 500 with overlap of 50 provided relevant context.
2. Smaller chunks improved precision but sometimes lost context.
3. FAISS enabled fast similarity search.
4. The RAG system generated grounded answers based on retrieved chunks.
5. Using custom PDFs allows answering questions on private documents.

# Conclusion

This project successfully implemented a Retrieval-Augmented Generation (RAG) system for document question answering.

The system loads custom PDF documents, converts them into embeddings, stores them in a FAISS vector database, retrieves relevant information, and generates context-aware answers using a local FLAN-T5 language model.

The approach improves factual accuracy and enables question answering over private documents.